# Module 3 — Batch Pipelines + PySpark Transformations
Exam domain: **Data Processing**

Runs standalone in Google Colab — no Databricks account needed.

In [ ]:
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

builder = (SparkSession.builder
    .appName("Module3-BatchPipelines")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## Source data
Two small tables: `orders` and `customers`.

In [ ]:
orders = spark.createDataFrame([
    (101, 1, "2024-02-01", 250.0),
    (102, 1, "2024-02-03", 80.0),
    (103, 2, "2024-02-02", 40.0),
    (104, 5, "2024-02-04", 10.0),   # customer_id 5 does not exist in customers
], ["order_id", "customer_id", "order_date", "amount"])

customers = spark.createDataFrame([
    (1, "Alice"), (2, "Bob"), (3, "Cara"),
], ["customer_id", "name"])

## Pure transformation functions
Writing transformations as pure functions (input DataFrame(s) -> output DataFrame)
makes them independently testable — this is the pattern used again in Module 8.

In [ ]:
def clean_orders(orders_df, min_order_date="2024-01-01"):
    return (orders_df
        .filter(F.col("amount") > 0)
        .filter(F.col("order_date") >= F.lit(min_order_date)))

def enrich_with_customer(orders_df, customers_df):
    return orders_df.join(customers_df, on="customer_id", how="left")

def add_running_total(df):
    w = Window.partitionBy("customer_id").orderBy("order_date")
    return df.withColumn("running_total", F.sum("amount").over(w))

In [ ]:
clean = clean_orders(orders, min_order_date="2024-01-01")
enriched = enrich_with_customer(clean, customers)
final_df = add_running_total(enriched)
final_df.orderBy("customer_id", "order_date").show()

## What happens with unmatched keys?
Order 104 references `customer_id = 5`, which doesn't exist in `customers`.
With a `left` join, `name` comes back `null` for that row — decide deliberately
whether that should be dropped, flagged, or sent to a quarantine table.

In [ ]:
final_df.filter(F.col("name").isNull()).show()

## Break it on purpose
Uncomment the line below to join on a column that doesn't exist and read the
full stack trace end to end — this is the debugging skill the checklist asks for.

In [ ]:
# broken = orders.join(customers, on="custommer_id", how="left")  # typo on purpose

## What if a source table is empty?

In [ ]:
empty_customers = customers.limit(0)
result = enrich_with_customer(clean, empty_customers)
result.show()  # every row should show name = null, and the join should not fail

## Orchestrating Bronze -> Silver -> Gold with these functions

In [ ]:
def run_pipeline(orders_df, customers_df, min_order_date="2024-01-01"):
    silver = enrich_with_customer(clean_orders(orders_df, min_order_date), customers_df)
    gold = (silver.groupBy("customer_id", "name")
                  .agg(F.sum("amount").alias("total_amount"),
                       F.count("*").alias("num_orders")))
    return silver, gold

silver_df, gold_df = run_pipeline(orders, customers)
gold_df.orderBy("customer_id").show()